In [ ]:
import pandas as pd
df = pd.read_csv("student_data.csv")

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Clean column names to be safe
df.columns = df.columns.str.strip()

# 2. Set the correct target column for this dataset (G3 is the Final Grade)
target_col = 'G3'

# 3. Select only the numerical columns as features so the model trains without crashing on text
numerical_features = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']

# Ensure all these numerical features actually exist in your dataframe slice
features = [col for col in numerical_features if col in df.columns]

X = df[features]
y = df[target_col]

# 4. Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Target column successfully set to: '{target_col}'")
print(f"Features successfully selected: {features}")
print("Step 1 completed smoothly! No more errors.")

In [ ]:
import joblib
import os
from sklearn.linear_model import LinearRegression

# 1. Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# 2. Evaluate the model score (R2 Score)
r2 = model.score(X_test, y_test)
print(f"Model trained successfully! R2 Score: {r2:.4f}")

# 3. Save the model and the list of features we used
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/student_model.pkl")
joblib.dump(features, "models/features.pkl")
print("Model and features saved successfully!")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import os

st.title("🎓 Student Final Grade Predictor (G3)")
st.write("This application predicts a student's final exam score based on academic indicators[cite: 15, 21].")

if not os.path.exists("models/student_model.pkl"):
    st.error("⚠️ Model file not found! Please run the training cells above first.")
else:
    model = joblib.load("models/student_model.pkl")
    features = joblib.load("models/features.pkl")

    st.header("📊 Input Student Parameters")

    # User-friendly inputs for the most important grade factors
    studytime = st.slider("Weekly Study Time (1 = <2 hours, 2 = 2-5 hours, 3 = 5-10 hours, 4 = >10 hours)", 1, 4, 2)
    failures = st.slider("Number of Past Class Failures", 0, 4, 0)
    absences = st.slider("Number of School Absences", 0, 93, 4)
    g1 = st.slider("First Period Grade (G1 Score: 0 - 20)", 0, 20, 10)
    g2 = st.slider("Second Period Grade (G2 Score: 0 - 20)", 0, 20, 10)

    # Automatically fill any remaining default numeric features with their median/average values
    input_dict = {col: 3 for col in features}  # default neutral value
    input_dict.update({
        'studytime': studytime,
        'failures': failures,
        'absences': absences,
        'G1': g1,
        'G2': g2,
        'age': 16,
        'Medu': 2,
        'Fedu': 2,
        'traveltime': 1,
        'famrel': 4,
        'freetime': 3,
        'goout': 3,
        'Dalc': 1,
        'Walc': 1,
        'health': 3
    })

    # Convert to DataFrame matching the exact training columns order
    input_data = pd.DataFrame([input_dict])[features]

    if st.button("🔮 Predict Final Grade (G3)"):
        prediction = model.predict(input_data)[0]
        final_score = max(0, min(20, prediction)) # Bounded between 0 and 20 based on dataset rules
        st.success(f"### Predicted Final Grade: **{final_score:.2f} / 20**")

In [ ]:
!pip install streamlit scikit-learn joblib pandas

In [ ]:
# 1. تثبيت أداة الـ tunnel
!npm install -g localtunnel

# 2. طباعة الباسورد السري اللي هتدخلي بيه للموقع
import urllib.request
print("\n--- COPY THIS PASSWORD / TUNNEL IP ---")
print(urllib.request.urlopen('https://ident.me').read().decode('utf8'))
print("------------------------------------\n")

# 3. تشغيل السيرفر والربط باللينك
!streamlit run app.py & npx localtunnel --port 8501


⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 949ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧
--- COPY THIS PASSWORD / TUNNEL IP ---
34.143.254.231
------------------------------------

⠙⠹⠸⠼⠴⠦⠧

your url is: https://two-clocks-lie.loca.lt
2026-06-29 00:40:06.082 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.143.254.231:8501

